# Construção dos indicadores

Nesta etapa são calculados os principais indicadores educacionais e escolares a partir da base analítica integrada, preparando as medidas que serão utilizadas nas análises e no dashboard.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED = PROJECT_ROOT / "data" / "processed"

base = pd.read_csv(
    PROCESSED / "base_analitica_escola_ano.csv",
    dtype={"codigo_escola": "string"}
)

kpis_gerais = {
    "taxa_aprovacao_media": base["taxa_aprovacao"].mean(),
    "taxa_reprovacao_media": base["taxa_reprovacao"].mean(),
    "taxa_abandono_media": base["taxa_abandono"].mean(),
    "media_alunos_turma": base["media_alunos_fund"].mean(),
    "matriculas_medias": base["QT_MAT_FUND"].mean(),
    "escolas_unicas": base["codigo_escola"].nunique(),
}

for indicador, valor in kpis_gerais.items():
    print(f"{indicador}: {valor:.2f}")

taxa_aprovacao_media: 89.45
taxa_reprovacao_media: 8.92
taxa_abandono_media: 1.64
media_alunos_turma: 20.73
matriculas_medias: 390.77
escolas_unicas: 276.00


## Indicadores por ano

Os principais KPIs são calculados por ano para acompanhar a evolução dos indicadores educacionais ao longo do período analisado.

In [2]:
kpis_ano = (
    base.groupby("ano")
    .agg(
        escolas=("codigo_escola", "nunique"),
        taxa_aprovacao=("taxa_aprovacao", "mean"),
        taxa_reprovacao=("taxa_reprovacao", "mean"),
        taxa_abandono=("taxa_abandono", "mean"),
        media_alunos_turma=("media_alunos_fund", "mean"),
        matriculas_medias=("QT_MAT_FUND", "mean"),
    )
    .reset_index()
)

display(kpis_ano.round(2))

,ano,escolas,taxa_aprovacao,taxa_reprovacao,taxa_abandono,media_alunos_turma,matriculas_medias
0,2018,276,79.97,19.05,0.98,21.09,398.24
1,2019,272,83.61,15.35,1.03,21.21,397.78
2,2020,271,99.46,0.13,0.41,21.22,394.27
3,2021,268,95.47,0.04,4.49,20.13,382.35
4,2022,267,89.16,9.27,1.57,20.54,391.43
5,2023,267,89.31,9.31,1.39,20.17,380.16


## Indicadores por dependência administrativa

Os principais KPIs são comparados entre as redes Federal, Estadual e Municipal para identificar diferenças no desempenho e no perfil das escolas.

In [3]:
kpis_dependencia = (
    base.groupby("dependencia_administrativa")
    .agg(
        escolas=("codigo_escola", "nunique"),
        taxa_aprovacao=("taxa_aprovacao", "mean"),
        taxa_reprovacao=("taxa_reprovacao", "mean"),
        taxa_abandono=("taxa_abandono", "mean"),
        media_alunos_turma=("media_alunos_fund", "mean"),
        matriculas_medias=("QT_MAT_FUND", "mean"),
    )
    .reset_index()
)

display(kpis_dependencia.round(2))

,dependencia_administrativa,escolas,taxa_aprovacao,taxa_reprovacao,taxa_abandono,media_alunos_turma,matriculas_medias
0,Estadual,220,89.12,9.05,1.83,19.90,328.56
1,Federal,2,95.65,4.35,0.00,26.14,387.67
2,Municipal,54,90.61,8.52,0.87,23.99,637.60


### Observações

A rede Municipal apresenta taxa média de aprovação ligeiramente superior à Estadual e concentra escolas de maior porte médio. A rede Federal apresenta os maiores valores médios de aprovação e alunos por turma, mas possui apenas duas escolas no recorte, o que exige cautela na comparação.

## Indicadores de infraestrutura

São calculados os percentuais de registros escola-ano que possuem determinados recursos de infraestrutura considerados comparáveis ao longo do período analisado.

In [4]:
variaveis_infraestrutura = {
    "IN_ESGOTO_REDE_PUBLICA": "Esgoto em rede pública",
    "IN_BIBLIOTECA": "Biblioteca",
    "IN_LABORATORIO_INFORMATICA": "Laboratório de informática",
    "IN_INTERNET": "Internet",
    "IN_BANDA_LARGA": "Banda larga",
    "IN_ACESSIBILIDADE_RAMPAS": "Rampas de acessibilidade",
}

kpis_infraestrutura = []

for coluna, descricao in variaveis_infraestrutura.items():
    dados_validos = base[coluna].dropna()

    percentual = (
        dados_validos.eq(1).mean() * 100
    )

    kpis_infraestrutura.append({
        "indicador": descricao,
        "percentual_com_recurso": percentual,
        "registros_validos": len(dados_validos),
    })

kpis_infraestrutura = pd.DataFrame(
    kpis_infraestrutura
)

display(
    kpis_infraestrutura.round(2)
)

,indicador,percentual_com_recurso,registros_validos
0,Esgoto em rede pública,96.05,1621
1,Biblioteca,83.78,1621
2,Laboratório de informática,74.83,1621
3,Internet,97.96,1621
4,Banda larga,83.26,1619
5,Rampas de acessibilidade,19.68,1621


### Observações

A maior parte dos registros possui acesso à internet, esgoto em rede pública, biblioteca e banda larga. A presença de rampas de acessibilidade é consideravelmente menor, aparecendo em cerca de 20% dos registros analisados.